# Aula 01 — Incerteza, espaço amostral, eventos e axiomas

## Objetivo

Construir eventos como conjuntos, verificar consequências dos axiomas em um espaço finito e comparar probabilidade teórica com frequência empírica em uma simulação reproduzível.

> Este notebook complementa a Aula 01 do módulo **Probabilidade, Estatística e Teoria da Informação**.

## Premissas e limites

- O dado é tratado como justo; por isso, as seis faces são equiprováveis.
- A moeda simulada tem probabilidade de cara fixada em `0.62`.
- A simulação usa um gerador pseudoaleatório e uma seed para permitir reprodução.
- Frequência empírica não é a própria probabilidade e simulação não substitui demonstração.
- A convergência formal será estudada na Aula 10, com a Lei dos Grandes Números.

## 1. Preparação

O Google Colab já inclui NumPy e Matplotlib. Em ambiente local, instale-os com `python -m pip install numpy matplotlib`.

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Matplotlib:', matplotlib.__version__)

## 2. Espaço amostral e eventos como conjuntos

No dado justo, defina $A$ como o evento “resultado par” e $B$ como “resultado pelo menos 4”. A função abaixo só usa contagem porque o modelo assume faces equiprováveis.

In [ ]:
omega = set(range(1, 7))
A = {2, 4, 6}
B = {4, 5, 6}

def prob_uniforme(evento, espaco):
    evento = set(evento)
    espaco = set(espaco)
    if not espaco:
        raise ValueError('O espaço amostral não pode ser vazio.')
    if not evento <= espaco:
        raise ValueError('O evento deve estar contido no espaço amostral.')
    return len(evento) / len(espaco)

eventos = {
    'A': A,
    'B': B,
    'A ∩ B': A & B,
    'A ∪ B': A | B,
    'Aᶜ': omega - A,
}

for nome, evento in eventos.items():
    print(f'{nome:5} = {sorted(evento)!s:18} P = {prob_uniforme(evento, omega):.3f}')

## 3. Verifique regras derivadas dos axiomas

Os testes confirmam, neste modelo, a regra do complemento, a regra geral da união e a monotonicidade.

In [ ]:
p_a = prob_uniforme(A, omega)
p_b = prob_uniforme(B, omega)
p_intersecao = prob_uniforme(A & B, omega)
p_uniao = prob_uniforme(A | B, omega)
p_complemento = prob_uniforme(omega - A, omega)

assert prob_uniforme(set(), omega) == 0
assert prob_uniforme(omega, omega) == 1
assert np.isclose(p_complemento, 1 - p_a)
assert np.isclose(p_uniao, p_a + p_b - p_intersecao)
assert p_intersecao <= p_a and p_intersecao <= p_b
print('Checks exatos concluídos: complemento, união e monotonicidade verificadas.')

## 4. Probabilidade teórica × frequência empírica

Simule 50 mil lançamentos de uma moeda com $P(\text{cara})=0.62$. A linha laranja é um parâmetro do modelo; a linha azul é calculada com os resultados observados até cada instante.

In [ ]:
SEED = 42
N = 50_000
P_CARA = 0.62

rng = np.random.default_rng(SEED)
caras = rng.random(N) < P_CARA
frequencia_acumulada = np.cumsum(caras) / np.arange(1, N + 1)

for n in [10, 100, 1_000, 10_000, 50_000]:
    print(f'n={n:>6}: frequência={frequencia_acumulada[n - 1]:.4f}')

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(np.arange(1, N + 1), frequencia_acumulada, color='#16b8f3',
        linewidth=1.3, label='frequência acumulada')
ax.axhline(P_CARA, color='#ffb347', linestyle='--', linewidth=2,
           label='probabilidade teórica = 0,62')
ax.set(xscale='log', xlabel='número de lançamentos (escala log)',
       ylabel='proporção de caras',
       title='Probabilidade teórica × frequência empírica')
ax.set_ylim(0.35, 0.85)
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## 5. Verifique eventos compostos por simulação

Agora estime as probabilidades de $A$, $B$, $A\cap B$ e $A\cup B$ com lançamentos simulados do dado.

In [ ]:
N_DADO = 120_000
rng_dado = np.random.default_rng(SEED + 1)
lancamentos = rng_dado.integers(1, 7, size=N_DADO)

a_obs = np.isin(lancamentos, list(A))
b_obs = np.isin(lancamentos, list(B))

estimativas = {
    'P̂(A)': a_obs.mean(),
    'P̂(B)': b_obs.mean(),
    'P̂(A ∩ B)': (a_obs & b_obs).mean(),
    'P̂(A ∪ B)': (a_obs | b_obs).mean(),
}

for nome, valor in estimativas.items():
    print(f'{nome:10} = {valor:.4f}')

lhs = estimativas['P̂(A ∪ B)']
rhs = estimativas['P̂(A)'] + estimativas['P̂(B)'] - estimativas['P̂(A ∩ B)']
assert np.isclose(lhs, rhs)
assert abs(estimativas['P̂(A)'] - 0.5) < 0.01
assert abs(estimativas['P̂(A ∪ B)'] - 2 / 3) < 0.01
assert abs(frequencia_acumulada[-1] - P_CARA) < 0.01
print('Checks empíricos concluídos: estimativas compatíveis com o modelo teórico.')

## 6. Desafios

1. Troque `P_CARA` por `0.50` e `0.20`. Compare as trajetórias.
2. Repita a moeda com seeds de 0 a 9. A aproximação ocorre pelo mesmo caminho?
3. Crie $C=\{1,2\}$ e verifique a monotonicidade para $C\subseteq B^c$.
4. Simule um dado enviesado com `rng.choice([1, 2, 3, 4, 5, 6], p=[...])`. Os pesos devem ser não negativos e somar 1.
5. Construa eventos disjuntos e confirme quando a soma simples pode ser usada.
6. Escreva duas frases: o que a simulação sustenta e o que ela não prova.

## Conclusões

- Resultados são elementos; eventos são subconjuntos do espaço amostral.
- A soma direta de probabilidades exige eventos disjuntos.
- Casos favoráveis sobre casos possíveis exige resultados equiprováveis.
- $P(A)$ pertence ao modelo; $\widehat p_n$ pertence à amostra observada.
- A seed ajuda na reprodução computacional, mas não é uma hipótese probabilística.

## Próxima etapa

Siga para a Aula 02 — **Contagem e combinatória para probabilidade**. Ela mostrará como contar configurações em espaços finitos sem enumerar cada possibilidade.